<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment43_PII_Detection_and_Redaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT 5
# PII DETECTION AND REDACTION
# Detection, Validation and Redaction of Personally
# Identifiable Information
# ============================================================

import re
from collections import Counter


# ============================================================
# 1. VERHOEFF CHECKSUM TABLES
# ============================================================

_D = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,2,3,4,0,6,7,8,9,5],
    [2,3,4,0,1,7,8,9,5,6],
    [3,4,0,1,2,8,9,5,6,7],
    [4,0,1,2,3,9,5,6,7,8],
    [5,9,8,7,6,0,4,3,2,1],
    [6,5,9,8,7,1,0,4,3,2],
    [7,6,5,9,8,2,1,0,4,3],
    [8,7,6,5,9,3,2,1,0,4],
    [9,8,7,6,5,4,3,2,1,0]
]

_P = [
    [0,1,2,3,4,5,6,7,8,9],
    [1,5,7,6,2,8,3,0,9,4],
    [5,8,0,3,7,9,6,1,4,2],
    [8,9,1,6,0,4,3,5,2,7],
    [9,4,5,3,1,2,6,8,7,0],
    [4,2,8,6,5,7,3,9,0,1],
    [2,7,9,3,8,0,6,4,1,5],
    [7,0,4,6,9,1,3,2,5,8]
]

_INV = [
    0,4,3,2,1,5,6,7,8,9
]


# ============================================================
# 2. VERHOEFF VALIDATION
# ============================================================

def verhoeff_valid(number):

    c = 0

    for i, ch in enumerate(
        reversed(str(number))
    ):

        if not ch.isdigit():
            return False

        c = _D[c][
            _P[i % 8][int(ch)]
        ]

    return c == 0


# ============================================================
# 3. GENERATE VERHOEFF CHECK DIGIT
# ============================================================

def verhoeff_checkdigit(number):

    c = 0

    for i, ch in enumerate(
        reversed(str(number))
    ):

        if not ch.isdigit():
            return None

        c = _D[c][
            _P[(i + 1) % 8][int(ch)]
        ]

    return _INV[c]


# ============================================================
# 4. LUHN VALIDATION
# ============================================================

def luhn_valid(number):

    digits = [
        int(d)
        for d in re.sub(
            r"\D",
            "",
            str(number)
        )
    ]

    if len(digits) < 12:
        return False

    total = 0
    parity = len(digits) % 2

    for i, digit in enumerate(digits):

        d = digit

        if i % 2 == parity:

            d *= 2

            if d > 9:
                d -= 9

        total += d

    return total % 10 == 0


# ============================================================
# 5. PII DETECTION PATTERNS
# ============================================================

PATTERNS = {

    "AADHAAR": (
        r"\b[2-9]\d{3}[ -]?\d{4}[ -]?\d{4}\b",

        lambda s:
        verhoeff_valid(
            re.sub(r"\D", "", s)
        )
    ),

    "PAN": (
        r"\b[A-Z]{5}\d{4}[A-Z]\b",

        lambda s: True
    ),

    "CARD": (
        r"\b(?:\d[ -]?){12,18}\d\b",

        lambda s:
        luhn_valid(s)
    ),

    "EMAIL": (
        r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",

        lambda s: True
    ),

    "PHONE_IN": (
        r"(?<!\d)(?:\+91[ -]?)?[6-9]\d{9}(?!\d)",

        lambda s: True
    ),

    "IPV4": (
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",

        lambda s:
        all(
            0 <= int(octet) <= 255
            for octet in s.split(".")
        )
    )
}


# ============================================================
# 6. DETECT VALIDATED PII
# ============================================================

def detect_pii(text):

    found = []

    for label, (
        pattern,
        validator
    ) in PATTERNS.items():

        for match in re.finditer(
            pattern,
            text
        ):

            value = match.group(0)

            if validator(value):

                found.append(
                    (
                        label,
                        value,
                        match.start(),
                        match.end()
                    )
                )


    # Sort by position and prefer longest match
    found.sort(
        key=lambda item:
        (
            item[2],
            -(item[3] - item[2])
        )
    )


    # Remove overlapping matches
    kept = []
    last_end = -1

    for item in found:

        if item[2] >= last_end:

            kept.append(item)
            last_end = item[3]


    return kept


# ============================================================
# 7. MASK OR REMOVE PII
# ============================================================

def redact(
    text,
    mode="mask"
):

    items = sorted(
        detect_pii(text),
        key=lambda item: -item[2]
    )

    output = text


    for label, value, start, end in items:

        if mode == "remove":

            replacement = (
                f"[{label}_REDACTED]"
            )

        else:

            replacement = (
                f"[{label}:****{value[-4:]}]"
            )


        output = (
            output[:start]
            + replacement
            + output[end:]
        )


    return output


# ============================================================
# 8. PII SUMMARY
# ============================================================

def summary(text):

    return dict(
        Counter(
            label
            for label, _, _, _
            in detect_pii(text)
        )
    )


# ============================================================
# 9. TEST CASES
# ============================================================

def run_tests():

    # --------------------------------------------------------
    # Generate synthetic Aadhaar-format number
    # --------------------------------------------------------

    aadhaar_base = "23456789012"

    aadhaar = (
        aadhaar_base
        +
        str(
            verhoeff_checkdigit(
                aadhaar_base
            )
        )
    )


    # Synthetic Luhn-valid test number
    card = "4539578763621486"


    # Sample synthetic complaint
    sample = (
        f"Complainant Ravi "
        f"(ravi.k@example.com, +91 9876543210) "
        f"states that his Aadhaar {aadhaar} "
        f"and PAN ABCDE1234F were used from host "
        f"192.168.1.55 to add card {card} to a wallet."
    )


    results = []


    # --------------------------------------------------------
    # TC1 - Aadhaar passes Verhoeff
    # --------------------------------------------------------

    results.append(
        (
            "TC1 synthetic Aadhaar passes Verhoeff",
            verhoeff_valid(aadhaar)
        )
    )


    # --------------------------------------------------------
    # TC2 - Corrupted Aadhaar fails
    # --------------------------------------------------------

    corrupted_digit = (
        str(
            (int(aadhaar[-1]) + 1)
            % 10
        )
    )

    corrupted_aadhaar = (
        aadhaar[:-1]
        +
        corrupted_digit
    )


    results.append(
        (
            "TC2 corrupted Aadhaar fails Verhoeff",
            not verhoeff_valid(
                corrupted_aadhaar
            )
        )
    )


    # --------------------------------------------------------
    # TC3 - Card passes Luhn
    # --------------------------------------------------------

    results.append(
        (
            "TC3 test card passes Luhn",
            luhn_valid(card)
        )
    )


    # --------------------------------------------------------
    # TC4 - Corrupted card fails Luhn
    # --------------------------------------------------------

    if card[-1] != "0":

        corrupted_card = (
            card[:-1] + "0"
        )

    else:

        corrupted_card = (
            card[:-1] + "1"
        )


    results.append(
        (
            "TC4 corrupted card fails Luhn",
            not luhn_valid(
                corrupted_card
            )
        )
    )


    # --------------------------------------------------------
    # TC5 - Detect all six PII types
    # --------------------------------------------------------

    detected_summary = summary(
        sample
    )


    all_types = all(
        detected_summary.get(
            key,
            0
        ) >= 1

        for key in [
            "AADHAAR",
            "PAN",
            "CARD",
            "EMAIL",
            "PHONE_IN",
            "IPV4"
        ]
    )


    results.append(
        (
            "TC5 all six PII types detected",
            all_types
        )
    )


    # --------------------------------------------------------
    # TC6 - Masked output hides original values
    # --------------------------------------------------------

    masked = redact(
        sample,
        "mask"
    )


    results.append(
        (
            "TC6 masked output hides raw values",
            aadhaar not in masked
            and
            card not in masked
            and
            "ravi.k@example.com"
            not in masked
        )
    )


    # --------------------------------------------------------
    # TC7 - Mask retains last four characters
    # --------------------------------------------------------

    results.append(
        (
            "TC7 masked output keeps last 4",
            f"****{aadhaar[-4:]}"
            in masked
        )
    )


    # --------------------------------------------------------
    # TC8 - Full redaction
    # --------------------------------------------------------

    removed = redact(
        sample,
        "remove"
    )


    results.append(
        (
            "TC8 remove mode redacts PII",
            "[AADHAAR_REDACTED]"
            in removed
            and
            "[PAN_REDACTED]"
            in removed
            and
            "[CARD_REDACTED]"
            in removed
        )
    )


    # --------------------------------------------------------
    # TC9 - Invalid IP rejected
    # --------------------------------------------------------

    invalid_ip_text = (
        "host 999.1.1.1 contacted us"
    )


    results.append(
        (
            "TC9 invalid IP rejected",
            "IPV4"
            not in summary(
                invalid_ip_text
            )
        )
    )


    # --------------------------------------------------------
    # TC10 - Clean text produces no findings
    # --------------------------------------------------------

    clean_text = (
        "The server rebooted at 04:00."
    )


    results.append(
        (
            "TC10 clean text yields no findings",
            summary(clean_text) == {}
        )
    )


    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("=" * 75)
    print("PII DETECTION AND REDACTION - TEST RESULTS")
    print("=" * 75)


    for name, passed in results:

        print(
            f"{name:<48} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )


    passed_count = sum(
        1
        for _, passed in results
        if passed
    )


    print("-" * 75)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    print("=" * 75)


    return (
        passed_count == len(results)
    )


# ============================================================
# 10. RUN TESTS
# ============================================================

run_tests()


# ============================================================
# 11. DISPLAY SAMPLE PII ANALYSIS
# ============================================================

print("\n")
print("=" * 75)
print("SAMPLE PII ANALYSIS")
print("=" * 75)


# Generate a fresh synthetic Aadhaar
aadhaar_base = "23456789012"

aadhaar = (
    aadhaar_base
    +
    str(
        verhoeff_checkdigit(
            aadhaar_base
        )
    )
)


card = "4539578763621486"


sample_text = (
    f"Contact ravi.k@example.com or +91 9876543210. "
    f"Aadhaar {aadhaar}, PAN ABCDE1234F, "
    f"IP address 192.168.1.55 and card {card}."
)


# ============================================================
# ORIGINAL
# ============================================================

print("\nORIGINAL:")
print(sample_text)


# ============================================================
# DETECTED PII
# ============================================================

print("\nDETECTED PII:")

findings = detect_pii(
    sample_text
)


for label, value, start, end in findings:

    print(
        f"{label:<12} : {value}"
    )


# ============================================================
# MASKED OUTPUT
# ============================================================

masked_output = redact(
    sample_text,
    "mask"
)


print("\nMASKED OUTPUT:")
print(masked_output)


# ============================================================
# FULLY REDACTED OUTPUT
# ============================================================

redacted_output = redact(
    sample_text,
    "remove"
)


print("\nFULLY REDACTED OUTPUT:")
print(redacted_output)


# ============================================================
# SUMMARY
# ============================================================

print("\nPII SUMMARY:")
print(summary(sample_text))


# ============================================================
# FINAL RESULT
# ============================================================

print("\n")
print("=" * 75)
print("EXPERIMENT 5 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    "PII was detected using pattern matching."
)

print(
    "Checksum validation was used to reduce false positives."
)

print(
    "Masked and fully redacted outputs were generated."
)

print("=" * 75)

PII DETECTION AND REDACTION - TEST RESULTS
TC1 synthetic Aadhaar passes Verhoeff            -> PASS
TC2 corrupted Aadhaar fails Verhoeff             -> PASS
TC3 test card passes Luhn                        -> PASS
TC4 corrupted card fails Luhn                    -> PASS
TC5 all six PII types detected                   -> PASS
TC6 masked output hides raw values               -> PASS
TC7 masked output keeps last 4                   -> PASS
TC8 remove mode redacts PII                      -> PASS
TC9 invalid IP rejected                          -> PASS
TC10 clean text yields no findings               -> PASS
---------------------------------------------------------------------------
RESULT: 10/10 test cases passed


SAMPLE PII ANALYSIS

ORIGINAL:
Contact ravi.k@example.com or +91 9876543210. Aadhaar 234567890124, PAN ABCDE1234F, IP address 192.168.1.55 and card 4539578763621486.

DETECTED PII:
EMAIL        : ravi.k@example.com
PHONE_IN     : +91 9876543210
AADHAAR      : 234567890124
PAN 